In [ ]:
import torch
from hcr import CRNN, HTRDataset, collate_fn, HTRTrainer, HTRInference
import os
import json
from torch.utils.data import DataLoader

import os
import json
import numpy as np
import cv2
import Levenshtein
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

from preprocessing_utils import preprocess_image

from doctr.io import DocumentFile
from doctr.models import detection_predictor
import cv2
import numpy as np

from IPython.display import display, clear_output
from pathlib import Path

Configuracion

In [ ]:
CHARSET = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
               "0123456789 $#&%*.,;:¡!¿?«»-_'\"()[]áéíóúüñÁÉÍÓÚÜÑ")
IMG_HEIGHT = 64          # altura de precomputación y entrada al modelo
IMG_WIDTH  = 256         # FIX 3: era 320 → 256  (suficiente para palabras)
BATCH_SIZE = 128         # FIX 5: era 96 → 128   (mejor saturación GPU con AMP)

# IMG_HEIGHT = 64 #48 #32   # ← captura mejor ascendentes/descendentes (f, g, p, y...)
# IMG_WIDTH  = 320 #768 #512 #256  # ← necesario si tus muestras son líneas de texto completa #3072 para iam
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# BATCH_SIZE = 64 if DEVICE == 'cuda' else 32 #128 #8 para iam
NUM_EPOCHS = 100
PIN_MEMORY = (DEVICE == 'cuda')

print(f"Dispositivo: {DEVICE}")
print(f"Caracteres en vocabulario: {len(CHARSET)}")

Instanciar modelo

In [ ]:
model = CRNN(
    num_classes=len(CHARSET),
    img_height=IMG_HEIGHT,
    hidden_channels=32,    # ← era 64
    lstm_hidden=128        # ← era 256
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales:    {total_params:,}")
print(f"Parámetros entrenables: {trainable:,}")

preprocesar conjunto train y guardarlo

In [ ]:
with open("dataset1/datos_ent_augmented/0annotation.json", "r", encoding="utf-8") as f:
    train = json.load(f)

images_dir = Path("dataset1/datos_ent_augmented")

image_paths = [
    images_dir / img_name
    for img_name in train.keys()
    if (images_dir / img_name).exists()
]

print('empezar precomputado')

train_dataset = HTRDataset(
    image_paths=image_paths,
    labels=train,
    charset=CHARSET,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH
)

train_dataset.precompute_chunked('chunks_train/', chunk_size=10000)

preprocesar conjunto validacion y guardarlo

In [ ]:
with open("dataset1/splits_test/val.json", "r", encoding="utf-8") as f:
    val = json.load(f)

images_dir = Path("dataset1/datos_testing")

image_paths = [
    images_dir / img_name
    for img_name in val.keys()
    if (images_dir / img_name).exists()
]

val_dataset = HTRDataset(
    image_paths= image_paths,
    labels=val,
    charset=CHARSET,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH
)

val_dataset.precompute_chunked('chunks_val/', chunk_size=10000)

cargar datasets

In [ ]:
#precomputar dataset
train_dataset = HTRDataset.from_precomputed('precomputed_train.pt', CHARSET)
val_dataset   = HTRDataset.from_precomputed('precomputed_val.pt',   CHARSET)

codificar las transcripciones

In [ ]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
    num_workers=4,
    persistent_workers=False,
    prefetch_factor=None,
    drop_last=True           # OK en train — protege BatchNorm
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
    num_workers=4,
    persistent_workers=False,
    prefetch_factor=None,
    drop_last=False          # IMPORTANTE — evalúa todas las muestras
)

print(f"Train: {len(train_dataset)} muestras")
print(f"Val:   {len(val_dataset)} muestras")

Entrenar por sesiones para no petar la pc aiuda

In [ ]:

trainer = HTRTrainer(
    model = model,
    charset=CHARSET,
    hidden_channels=512,
    lstm_hidden=512,
    device=DEVICE,
    lr=3e-4
)

In [ ]:
trainer.plot_metrics(save_path='curvas_epoch100.png')

In [ ]:
trainer.load_checkpoint('checkpoints/best_model.pt')

In [ ]:

IMAGE_FOLDER='test_informe'
JSON_PATH='test_informe/test.json'

# IMAGE_FOLDER = 'pinterest/seg'
# JSON_PATH = 'pinterest/split/content_test.json'

with open(JSON_PATH, "r", encoding="utf-8") as f:
    el_json = json.load(f)

inferencer = HTRInference(
    model_path='checkpoints/ver10_from0_again2/best_model.pt',
    charset=CHARSET,
    device=DEVICE,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    hidden_channels=32,
    lstm_hidden=128,
)

image_entries = []

for entry in sorted(os.scandir(IMAGE_FOLDER), key=lambda e: e.name):
    if entry.is_file():
        # Caso plano: IMAGE_FOLDER contiene las imágenes directamente
        # Clave candidata: "imagen.png" o "imagen"
        image_entries.append((entry.name, entry.path))
    elif entry.is_dir():
        # Caso anidado: IMAGE_FOLDER contiene subcarpetas
        # Clave candidata: "subcarpeta/imagen.png" o "subcarpeta/imagen"
        for sub_entry in sorted(os.scandir(entry.path), key=lambda e: e.name):
            if sub_entry.is_file():
                relative = f"{entry.name}/{sub_entry.name}"
                image_entries.append((relative, sub_entry.path))

for key_with_ext, image_path in image_entries:
    key_without_ext = os.path.splitext(key_with_ext)[0]

    if key_with_ext in el_json:
        clave = key_with_ext
    elif key_without_ext in el_json:
        clave = key_without_ext
    else:
        continue

    filename   = os.path.basename(image_path)
    prediction = inferencer.predict(image_path, use_beam_search=True)
    reference  = el_json[clave]

# filename   = 'photo_2026-05-11_12-17-23.jpg'
# prediction = inferencer.predict(filename, use_beam_search=True)
# reference  = 'el modelo tiene que funcionar bien'

    print(f"  Imagen:        {filename}")
    print(f"  Etiqueta real: {reference}")
    print(f"  Predicción:    {prediction}")
    print("-" * 40)

In [ ]:

IMAGE_FOLDER='FINAL/final'
JSON_PATH='FINAL/split_final/test.json'

with open(JSON_PATH, "r", encoding="utf-8") as f:
    el_json = json.load(f)

inferencer = HTRInference(
    model_path='checkpoints/ver10_from0_again2/best_model.pt',
    charset=CHARSET,
    device=DEVICE,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    hidden_channels=32,
    lstm_hidden=128,
)

def cer(predictions, targets):
    """
    CER = Character Error Rate
    CER = edit_distance(pred, target) / len(target)

    Mide el % de caracteres que hay que insertar/borrar/sustituir
    para transformar la predicción en el texto correcto.
    """
    total_edit = 0
    total_chars = 0

    for pred, target in zip(predictions, targets):
        edit_dist = edit_distance(pred, target)
        total_edit += edit_dist
        total_chars += len(target)

    return total_edit / max(total_chars, 1)

def wer(predictions, targets):
    """
    WER = Word Error Rate
    Similar al CER pero a nivel de palabras.
    """
    total_edit = 0
    total_words = 0

    for pred, target in zip(predictions, targets):
        pred_words = pred.split()
        target_words = target.split()
        edit_dist = edit_distance(pred_words, target_words)
        total_edit += edit_dist
        total_words += len(target_words)

    return total_edit / max(total_words, 1)

def edit_distance(s1, s2):
    """Distancia de Levenshtein entre dos secuencias."""
    m, n = len(s1), len(s2)
    dp = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],    # borrar
                                       dp[i][j-1],    # insertar
                                       dp[i-1][j-1])  # sustituir
    return dp[m][n]

def preprocess_image2(img_path):
    versions = preprocess_image(image_path=img_path)
    h, w = versions[5].shape
    new_w = max(int(w * IMG_HEIGHT / h), 1)
    img = cv2.resize(versions[5], (new_w, IMG_HEIGHT), interpolation=cv2.INTER_LINEAR)
    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w), dtype=np.uint8) * 255
        img = np.hstack([img, pad])
    else:
        img = img[:, :IMG_WIDTH]
    return img

def get_char_pairs(reference: str, hypothesis: str):
    """Alinea ref e hyp con Levenshtein y devuelve pares (real, predicho)."""
    ops     = Levenshtein.editops(reference, hypothesis)
    ref_lst = list(reference)
    hyp_lst = list(hypothesis)
    pairs   = []
    used_ref, used_hyp = set(), set()

    for op, i, j in ops:
        if op == 'replace':
            pairs.append((ref_lst[i], hyp_lst[j]))
            used_ref.add(i); used_hyp.add(j)
        elif op == 'delete':
            pairs.append((ref_lst[i], '<DEL>'))
            used_ref.add(i)
        elif op == 'insert':
            pairs.append(('<INS>', hyp_lst[j]))
            used_hyp.add(j)

    for i, c in enumerate(ref_lst):   # caracteres correctos
        if i not in used_ref:
            pairs.append((c, c))

    return pairs

def bootstrap_cer_ci(refs, hyps, n_bootstrap=2000, ci=95):
    """IC por bootstrap sobre el CER global."""
    n = len(refs)
    scores = []
    for _ in range(n_bootstrap):
        idx      = np.random.choice(n, n, replace=True)
        s_refs   = [refs[i] for i in idx]
        s_hyps   = [hyps[i] for i in idx]
        scores.append(cer(s_refs, s_hyps))
    lo = (100 - ci) / 2
    return np.percentile(scores, [lo, 100 - lo])

def plot_confusion_matrix(confusion: dict, top_n: int = 30, save_path="confusion_matrix.png"):
    """Dibuja la matriz de confusión de los top_n caracteres más frecuentes."""
    # Caracteres más frecuentes (excluye etiquetas especiales para el eje)
    char_counts = defaultdict(int)
    for real, preds in confusion.items():
        for count in preds.values():
            char_counts[real] += count

    top_chars = [c for c, _ in sorted(char_counts.items(), key=lambda x: -x[1])
                 if c not in ('<DEL>', '<INS>')][:top_n]

    all_labels = top_chars + ['<DEL>', '<INS>']
    label_idx  = {l: i for i, l in enumerate(all_labels)}
    size       = len(all_labels)
    matrix     = np.zeros((size, size), dtype=int)

    for real, preds in confusion.items():
        if real not in label_idx:
            continue
        for pred, count in preds.items():
            if pred in label_idx:
                matrix[label_idx[real], label_idx[pred]] += count

    fig, ax = plt.subplots(figsize=(max(10, size // 2), max(8, size // 2)))
    sns.heatmap(
        matrix, annot=True, fmt='d', cmap='Blues',
        xticklabels=all_labels, yticklabels=all_labels,
        linewidths=0.3, ax=ax, cbar_kws={'label': 'Frecuencia'}
    )
    ax.set_xlabel("Predicho");  ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusión por carácter (top {top_n})")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"[✓] Matriz guardada en '{save_path}'")

def print_top_confusions(confusion: dict, top_n: int = 20):
    """Imprime los errores de sustitución más frecuentes."""
    errors = [
        (real, pred, count)
        for real, preds in confusion.items()
        for pred, count in preds.items()
        if real != pred
    ]
    errors.sort(key=lambda x: -x[2])
    print(f"\n{'─'*40}")
    print(f"  Top {top_n} confusiones de carácter")
    print(f"{'─'*40}")
    for real, pred, count in errors[:top_n]:
        bar = '█' * min(count, 40)
        print(f"  '{real}' → '{pred}': {count:4d}  {bar}")

all_refs  = []
all_hyps  = []
confusion = defaultdict(lambda: defaultdict(int))

image_entries = []

for entry in sorted(os.scandir(IMAGE_FOLDER), key=lambda e: e.name):
    if entry.is_file():
        # Caso plano: IMAGE_FOLDER contiene las imágenes directamente
        # Clave candidata: "imagen.png" o "imagen"
        image_entries.append((entry.name, entry.path))
    elif entry.is_dir():
        # Caso anidado: IMAGE_FOLDER contiene subcarpetas
        # Clave candidata: "subcarpeta/imagen.png" o "subcarpeta/imagen"
        for sub_entry in sorted(os.scandir(entry.path), key=lambda e: e.name):
            if sub_entry.is_file():
                relative = f"{entry.name}/{sub_entry.name}"
                image_entries.append((relative, sub_entry.path))

for key_with_ext, image_path in image_entries:
    key_without_ext = os.path.splitext(key_with_ext)[0]

    if key_with_ext in el_json:
        clave = key_with_ext
    elif key_without_ext in el_json:
        clave = key_without_ext
    else:
        continue

    filename   = os.path.basename(image_path)
    prediction = inferencer.predict(image_path, use_beam_search=True)
    reference  = el_json[clave]

    print(f"  Imagen:        {filename}")
    print(f"  Etiqueta real: {reference}")
    print(f"  Predicción:    {prediction}")
    print("-" * 40)

    all_refs.append(reference)
    all_hyps.append(prediction)

    for real, pred in get_char_pairs(reference, prediction):
        confusion[real][pred] += 1

total      = len(all_refs)
global_cer = cer(all_refs, all_hyps)
global_wer = wer(all_refs, all_hyps)
exact      = sum(r == h for r, h in zip(all_refs, all_hyps))
exact_pct  = exact / total * 100
ci_lo, ci_hi = bootstrap_cer_ci(all_refs, all_hyps)

print("\n" + "═" * 45)
print("  RESULTADOS FINALES")
print("═" * 45)
print(f"  Muestras evaluadas : {total}")
print(f"  CER                : {global_cer:.4f}  ({global_cer*100:.2f}%)")
print(f"  CER IC 95%%         : [{ci_lo*100:.2f}%, {ci_hi*100:.2f}%]")
print(f"  WER                : {global_wer:.4f}  ({global_wer*100:.2f}%)")
print(f"  Exact Match        : {exact}/{total}  ({exact_pct:.2f}%)")
print("═" * 45)

print_top_confusions(confusion, top_n=20)
plot_confusion_matrix(confusion, top_n=30, save_path="confusion_matrix.png")

In [ ]:
def sort_words_reading_order(words, line_tolerance=10):
    """
    Ordena words de izquierda a derecha y de arriba hacia abajo.
    line_tolerance: margen en píxeles para considerar que dos words están en la misma línea.
    """
    return sorted(words, key=lambda w: (
        round(w['bbox'][1] / line_tolerance),  # y1 agrupado por líneas
        w['bbox'][0]                            # x1 dentro de cada línea
    ))

def segment_words_doctr(image_path):
    model = detection_predictor(
        arch='db_resnet50',
        pretrained=True,
        assume_straight_pages=True
    )
    
    doc = DocumentFile.from_images(image_path)
    result = model(doc)  # result es una lista de dicts
    
    # Obtener dimensiones reales de la imagen
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    
    words = []
    for page in result:  # iterar la lista directamente
        word_array = page['words']  # array numpy [x1, y1, x2, y2, conf]
        for word in word_array:
            x1, y1, x2, y2, conf = word
            words.append({
                'bbox': (
                    int(x1 * w), int(y1 * h),
                    int(x2 * w), int(y2 * h)
                ),
                'confidence': float(conf)
            })
    
    words = sort_words_reading_order(words, line_tolerance=10)
    return words

def visualize_words(image_path, words, min_confidence=0.3):
    img = cv2.imread(image_path)
    
    for word in words:
        if word['confidence'] >= min_confidence:
            x1, y1, x2, y2 = word['bbox']
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    return img

def plot_words(image_path, words, min_confidence=0.3):
    img = visualize_words(image_path, words, min_confidence)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(15, 20))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'Palabras detectadas: {len(words)} (conf >= {min_confidence})')
    plt.tight_layout()
    plt.show()

In [ ]:
#primera sesion
trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=5
)

In [ ]:
#segunda sesion

#precomputar dataset
train_dataset = HTRDataset.from_precomputed('precomputed_train.pt', CHARSET)
val_dataset   = HTRDataset.from_precomputed('precomputed_val.pt',   CHARSET)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4
)

print(f"Train: {len(train_dataset)} muestras")
print(f"Val:   {len(val_dataset)} muestras")

#instanciar modelo de nuevo
model = CRNN(
    num_classes=len(CHARSET),
    img_height=IMG_HEIGHT,
    hidden_channels=32,   # reducir a 32 si tienes poca VRAM
    lstm_hidden=256
)

#instanciar trainer
trainer = HTRTrainer(
    model=model,
    charset=CHARSET,
    device=DEVICE,
    lr=1e-4
)

#cargar ultimo checkpoint
trainer.load_checkpoint('checkpoints/checkpoint_epoch030_cer0.85.pt')

#entrenar
trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=10
)
# trainer.plot_metrics(save_path='curvas_sesion2.png')    # muestra épocas 1-60 completas AL FINAL GRAFICAR TODE

4. Orden del "Curriculum Learning"Para que el modelo sea robusto, te sugiero este orden de entrenamiento:Fase 1 (Pre-train): Dataset sintético + Palabras/Letras sueltas. El modelo aprende la forma de los caracteres "perfectos". $LR = 3 \times 10^{-4}$.Fase 2 (Adaptación): Dataset aumentado (tus datos originales con ruido). Aquí el modelo aprende a lidiar con binarizaciones imperfectas. $LR = 1 \times 10^{-4}$.Fase 3 (Fine-tuning): IAM y manuscritos reales. El modelo aprende ligaduras y contexto de oraciones. $LR = 5 \times 10^{-5}$.

In [ ]:
def load_history(metrics_path):
    """Carga el historial de métricas si existe, si no devuelve uno vacío."""
    if os.path.exists(metrics_path):
        data = torch.load(metrics_path, weights_only=False)
        print(f"  Historial cargado: {len(data['loss_train'])} épocas previas")
        return data
    return {
        'epoch':      [],
        'loss_train': [],
        'loss_val':   [],
        'cer':        [],
        'wer':        [],
    }

def plot_metrics(history, save_path=None):
    """
    Grafica las curvas de loss y CER/WER usando el historial acumulado
    de todas las sesiones de entrenamiento.
    """
    if not history['epoch']:
        print("No hay métricas registradas todavía.")
        return

    epochs     = history['epoch']
    loss_train = history['loss_train']
    loss_val   = history['loss_val']
    cer        = history['cer']
    wer        = history['wer']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Curvas de entrenamiento', fontsize=14, fontweight='bold')

    ax1.plot(epochs, loss_train, label='Loss train', color='steelblue',   linewidth=2)
    ax1.plot(epochs, loss_val,   label='Loss val',   color='tomato',      linewidth=2)
    ax1.set_xlabel('Época')
    ax1.set_ylabel('CTC Loss')
    ax1.set_title('Loss train vs validación')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Marcar mínimo de val loss
    min_val_idx = int(np.argmin(loss_val))
    ax1.axvline(x=epochs[min_val_idx], color='tomato', linestyle='--',
                alpha=0.5, label=f'Min val época {epochs[min_val_idx]}')
    ax1.scatter([epochs[min_val_idx]], [loss_val[min_val_idx]],
                color='tomato', zorder=5, s=80)

    ax2.plot(epochs, cer, label='CER', color='darkorange', linewidth=2)
    ax2.plot(epochs, wer, label='WER', color='purple',     linewidth=2)
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Tasa de error')
    ax2.set_title('CER y WER')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1.05)

    # Marcar mínimo de CER
    min_cer_idx = int(np.argmin(cer))
    ax2.axvline(x=epochs[min_cer_idx], color='darkorange', linestyle='--',
                alpha=0.5)
    ax2.scatter([epochs[min_cer_idx]], [cer[min_cer_idx]],
                color='darkorange', zorder=5, s=80,
                label=f'Min CER={cer[min_cer_idx]:.4f} (época {epochs[min_cer_idx]})')
    ax2.legend()

    # Separar sesiones con línea vertical si hay saltos en las épocas
    for i in range(1, len(epochs)):
        if epochs[i] != epochs[i-1] + 1:
            ax1.axvline(x=epochs[i] - 0.5, color='gray',
                        linestyle=':', alpha=0.6, linewidth=1.5)
            ax2.axvline(x=epochs[i] - 0.5, color='gray',
                        linestyle=':', alpha=0.6, linewidth=1.5)
            ax1.text(epochs[i] - 0.5, ax1.get_ylim()[1] * 0.95,
                        'nueva\nsesión', fontsize=7, color='gray', ha='center')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Gráfica guardada en: {save_path}")

    plt.show()       

In [ ]:
history = load_history('checkpoints/ver2/metrics_history.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver3/metrics_history.pt')
plot_metrics(history)

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("TheFinAI/MultiFinBen-SpanishOCR")

In [ ]:
history = load_history('checkpoints/ver4_from0/metrics_history.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver5_from0/metrics_history_2.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver5_from0/metrics_history.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver4_from0/metrics_history_2.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver5_from0/metrics_history_2.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver5_from0/metrics_history_3.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver6_from0/metrics_history.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver6_from0/metrics_history_2.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver6_from0/metrics_history_3.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver6_from0/metrics_history_4.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver6_from0/metrics_history_5.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver7_from0_again/metrics_history.pt')
plot_metrics(history)

In [ ]:
history = load_history('checkpoints/ver7_from0_again/metrics_history_2.pt')
plot_metrics(history)

In [ ]:
from plot_history import load_history, plot_metrics

history = load_history('checkpoints/ver8_from0_again2/metrics_history.pt')
plot_metrics(history)

In [ ]:
from plot_history import plot_all_for_thesis

history = load_history('checkpoints/ver9_from0_again2/metrics_history.pt')
plot_all_for_thesis(history, 'plots_etapa2')

In [ ]:
history = load_history('checkpoints/ver10_from0_again2/metrics_history.pt')
plot_all_for_thesis(history, 'plots_etapa3')